# Predict lipophilicity: a notebook you can inspect and change

**Mode: editable student exercise on public data.** This notebook trains new
models in your own runtime. It never reads or overwrites the author's frozen
FreeSolv or ChEMBL EGFR results. No reference model scores are supplied yet.
An executed copy is your exercise, not an independent benchmark or discovery.

**Question:** can eight physical descriptors or a molecular fingerprint
predict measured logD at pH 7.4 for molecules with unseen scaffolds?
You will learn to inspect molecules, separate fitting from selection, compare
three baselines, and explain errors without tuning on the test set.
Prerequisites: basic Python, arrays, train/validation/test, and regression.

**Colab:** download this `.ipynb`, open [Colab](https://colab.research.google.com/),
sign in and choose **File → Upload notebook**. Save your own copy. Choose a
CPU runtime; this exercise does not need a GPU or API key. Local Jupyter is
equally supported. No Drive mount is required. Runtime files are temporary:
download your results before disconnecting. Free compute is not guaranteed;
see the [official FAQ](https://research.google.com/colaboratory/faq.html).

Run cells in order. After reading test errors, changes belong in a new
exploratory copy and cannot count as a fresh test of the original recipe.
Small-fixture engineering execution and real-assay results are different.

## 1. Prepare only the libraries we need

This cell installs missing or mismatched pinned dependencies into your
current Python environment. In Colab that is your disposable runtime;
for local Jupyter, first select a dedicated project kernel. Python 3.11+
is required; the local reference environment is Python 3.11.
The cell checks imported versions as well as installed package metadata.
If an older library is still in memory, restart the session as instructed
and run from the top. Installation failures are not a reason to silently
change versions. Actual Colab execution has not yet been verified.

In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

# This standalone notebook also runs outside the project's Python constraints.
if sys.version_info < (3, 11):  # noqa: UP036
    raise RuntimeError("Use Python 3.11 or newer")
required = {
    "numpy": "2.4.6", "pandas": "2.3.3", "rdkit": "2025.9.6",
    "scikit-learn": "1.9.0", "matplotlib": "3.11.1",
}


def version_key(value):
    return tuple(int(part) if part.isdigit() else part for part in str(value).split("."))


missing = []
loaded = False
for package, wanted in required.items():
    module_name = "sklearn" if package == "scikit-learn" else package
    module = sys.modules.get(module_name)
    imported_version = getattr(module, "__version__", "unknown")
    if module is not None and version_key(imported_version) != version_key(wanted):
        raise RuntimeError(
            f"{package} {imported_version} is already loaded; expected {wanted}. "
            "Restart the session, then Run all."
        )
    try:
        mismatch = version(package) != wanted
    except PackageNotFoundError:
        mismatch = True
    if mismatch:
        missing.append(f"{package}=={wanted}")
        loaded |= module is not None
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if loaded:
        raise RuntimeError("Dependencies changed: restart the session, then Run all")
print("Python", sys.version.split()[0])
print({package: version(package) for package in required})

## 2. Choose your working files

The default CSV is downloaded into the current runtime only when absent.
An existing named CSV is reused and validated, never silently replaced.
Each full execution creates a new output directory beneath the notebook's
current working directory, where the notebook file browser can find it. Re-executing
export cells updates files inside that directory. Download those outputs
if you want to keep them; none is written into another lab's results.

In [ ]:
import csv
import json
import math
import tempfile
import warnings
from collections import Counter, defaultdict
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

For your own local data copy, edit `DATA_PATH`. Keep the exact source and
structure/endpoint contract. `DATA_KIND` labels the exported report; it
does not transform invented labels into measurements.

In [ ]:
DATA_PATH = Path("Lipophilicity.csv")
DATA_KIND = "public-MoleculeNet-lipophilicity"
DATA_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv"
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="aidd-lipophilicity-", dir=Path.cwd()))

## 3. Inspect the endpoint before building a model

Source: [AstraZeneca deposit](https://doi.org/10.6019/CHEMBL3301361),
[assay CHEMBL3301363](https://www.ebi.ac.uk/chembl/api/data/assay/CHEMBL3301363.json),
and the CSV distributed by the [DeepChem 2.8.0 loader](https://github.com/deepchem/deepchem/blob/2.8.0/deepchem/molnet/load_function/lipo_datasets.py).
ChEMBL data are [CC BY-SA 3.0](https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/LICENSE);
attribute the deposit and curation. Data are downloaded, not bundled here.

`exp` measures logD at pH 7.4, not logP, binding affinity, or solubility.
The dated intake on 9 September 2026 contained 4,200 rows (0.283 MB), all
parseable. The server reported a 10 July 2020 modification date; this URL
is not immutable. Current ChEMBL records were not joined row by row to this
historical CSV. Keep the original structure strings, including water
fragments; no target clipping or error-based exclusions.

**Pause:** what would go wrong if the target were interpreted as pChEMBL?
The unit, physical meaning and useful comparisons would all change.

In [ ]:
def inspect(source: Path) -> dict:
    ids = set()
    structures = defaultdict(list)
    scaffolds = Counter()
    targets = []
    multicomponent_rows = 0
    with source.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["CMPD_CHEMBLID", "exp", "smiles"]:
            raise ValueError("Expected columns CMPD_CHEMBLID,exp,smiles in that order")
        for line, row in enumerate(reader, start=2):
            if None in row or any(value is None or not value.strip() for value in row.values()):
                raise ValueError(f"Line {line}: missing value or unexpected column")
            compound_id = row["CMPD_CHEMBLID"].strip()
            if compound_id in ids:
                raise ValueError(f"Line {line}: repeated compound ID {compound_id}")
            target = float(row["exp"])
            if not math.isfinite(target):
                raise ValueError(f"Line {line}: target must be finite")
            molecule = Chem.MolFromSmiles(row["smiles"])
            if molecule is None or molecule.GetNumAtoms() == 0:
                raise ValueError(f"Line {line}: invalid or empty molecule")
            structure = Chem.MolToSmiles(molecule, isomericSmiles=True)
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=molecule, includeChirality=False)
            ids.add(compound_id)
            targets.append(target)
            structures[structure].append(target)
            scaffolds[scaffold] += 1
            multicomponent_rows += len(Chem.GetMolFrags(molecule)) > 1
    if not targets:
        raise ValueError("Input contains no data rows")
    return {
        "rows": len(targets),
        "unique_ids": len(ids),
        "unique_canonical_isomeric_smiles": len(structures),
        "repeated_structure_groups": sum(len(values) > 1 for values in structures.values()),
        "repeated_structure_extra_rows": len(targets) - len(structures),
        "structure_groups_with_different_labels": sum(
            len(set(values)) > 1 for values in structures.values()
        ),
        "nonempty_scaffold_groups": sum(bool(scaffold) for scaffold in scaffolds),
        "largest_nonempty_scaffold_rows": max(
            (count for scaffold, count in scaffolds.items() if scaffold), default=0
        ),
        "acyclic_rows": scaffolds[""],
        "multicomponent_rows": multicomponent_rows,
        "target_range": [min(targets), max(targets)],
        "rdkit_version": rdBase.rdkitVersion,
        "scope": "Read-only intake; no rows excluded, no split assigned, no model fitted",
    }

def load_frame(path):
    """Validate without filtering, then add label-free structure and scaffold keys."""
    intake = inspect(Path(path))
    if intake["repeated_structure_groups"]:
        raise ValueError("Repeated structure: resolve the entity contract before modelling")
    frame = pd.read_csv(path)
    molecules = [Chem.MolFromSmiles(smiles) for smiles in frame["smiles"]]
    frame["canonical"] = [Chem.MolToSmiles(mol, isomericSmiles=True) for mol in molecules]
    frame["scaffold"] = [
        MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False) for mol in molecules
    ]
    return frame


if not DATA_PATH.exists():
    with urlopen(DATA_URL, timeout=60) as response:
        raw_csv = response.read()
    with DATA_PATH.open("xb") as handle:
        handle.write(raw_csv)
frame = load_frame(DATA_PATH)
print("Input kind:", DATA_KIND)
display(pd.Series(inspect(DATA_PATH), name="intake"))
display(frame.head())

## 4. Look at molecules, not only arrays

Each drawing is one supplied molecular record. Disconnected fragments
remain visible; no hidden salt stripping is performed. Multiple records
can share chemistry, so row count is not experimental replication.
Look for rings, heteroatoms and flexible linkers. Which might affect
membrane partitioning, and which effects depend on pH?

In [ ]:
display(Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in frame.smiles.head(6)],
    legends=frame.CMPD_CHEMBLID.head(6).tolist(),
    molsPerRow=3, subImgSize=(240, 180), useSVG=True,
))

## 5. Split by scaffold before fitting

We follow [DeepChem 2.8.0 scaffold ordering](https://github.com/deepchem/deepchem/blob/2.8.0/deepchem/splits/splitters.py):
larger groups first, then larger first-row index for ties. Requested
train/validation/test fractions are 80/10/10. Empty scaffolds stay in one
group. RDKit version and row order matter; this is not a random split.
Group-preserving row fractions may differ from the requested fractions.

Train learns coefficients and preprocessing. Validation chooses alpha.
Test evaluates the selected recipe. The assertions check complete assignment
and no ID, exact-structure or scaffold overlap. They do not prove independent
experimental batches or dissimilarity under every molecular representation.

In [ ]:
def validate_split(frame, splits):
    """Check complete assignment and compound/structure/scaffold separation."""
    if set(splits) != {"train", "validation", "test"}:
        raise ValueError("Expected train, validation, test")
    if any(len(indices) == 0 for indices in splits.values()):
        raise ValueError("Every split must contain rows; this grouping is not scorable")
    assigned = np.concatenate(list(splits.values()))
    if sorted(assigned.tolist()) != list(range(len(frame))):
        raise ValueError("Rows must appear exactly once across splits")
    for key in ["CMPD_CHEMBLID", "canonical", "scaffold"]:
        seen = set()
        for indices in splits.values():
            keys = set(frame.iloc[indices][key])
            if seen & keys:
                raise ValueError(f"Cross-split {key} overlap")
            seen.update(keys)

def scaffold_split(frame):
    """DeepChem 2.8.0 group-size/first-index ordering, requested 80/10/10.

    Empty scaffolds form one group. No target value participates in assignment.
    The algorithm follows the cited splitter; RDKit version still affects keys.
    """
    groups = defaultdict(list)
    for index, key in enumerate(frame["scaffold"]):
        groups[key].append(index)
    ordered = sorted(groups.values(), key=lambda group: (len(group), group[0]), reverse=True)
    splits = {"train": [], "validation": [], "test": []}
    for group in ordered:
        if len(splits["train"]) + len(group) <= 0.8 * len(frame):
            splits["train"].extend(group)
        elif len(splits["train"]) + len(splits["validation"]) + len(group) <= 0.9 * len(frame):
            splits["validation"].extend(group)
        else:
            splits["test"].extend(group)
    splits = {name: np.asarray(indices, dtype=int) for name, indices in splits.items()}
    validate_split(frame, splits)
    return splits


splits = scaffold_split(frame)
train, validation, test = (splits[name] for name in ("train", "validation", "test"))
counts = {name: len(indices) for name, indices in splits.items()}
display(pd.DataFrame([
    {"split": name, "rows": len(indices),
     "scaffold_groups": frame.iloc[indices].scaffold.nunique()}
    for name, indices in splits.items()
]))
membership = frame[["CMPD_CHEMBLID", "canonical", "scaffold"]].copy()
membership["split"] = ""
for name, indices in splits.items():
    membership.loc[indices, "split"] = name
membership.to_csv(OUTPUT_DIR / "split-membership.csv", index=False)

## 6. Two representations of the same molecules

The eight descriptor columns are molecular weight, MolLogP, polar surface
area, H-bond donors and acceptors, ring count, rotatable bonds and fraction
of sp3 carbons. **MolLogP is a feature, not a measured logD prediction.**
Morgan encodes radius-2 local neighbourhoods in 2,048 bits, without chirality.
Collisions are possible and 2D fingerprints do not describe assay conditions.

Features do not read labels. Computing a fixed per-molecule fingerprint
before partitioning its matrix is different from fitting a scaler on all rows.
The next step fits every scaler on training rows only.

In [ ]:
def molecular_features(smiles):
    """Eight interpretable descriptors and nonchiral radius-2 Morgan bit vectors."""
    molecules = [Chem.MolFromSmiles(value) for value in smiles]
    if not molecules or any(mol is None or mol.GetNumAtoms() == 0 for mol in molecules):
        raise ValueError("Features require valid nonempty molecules")
    names = [
        "MolWt",
        "MolLogP",
        "TPSA",
        "NumHDonors",
        "NumHAcceptors",
        "RingCount",
        "NumRotatableBonds",
        "FractionCSP3",
    ]
    descriptors = np.asarray(
        [[getattr(Descriptors, name)(mol) for name in names] for mol in molecules], dtype=float
    )
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048,
        includeChirality=False,
    )
    fingerprints = [generator.GetFingerprint(mol) for mol in molecules]
    bits = np.asarray([np.asarray(fp) for fp in fingerprints], dtype=float)
    if not np.isfinite(descriptors).all() or not np.isfinite(bits).all():
        raise ValueError("Non-finite feature: no silent imputation or exclusion")
    return descriptors, bits, fingerprints


descriptors, morgan, fingerprints = molecular_features(frame.smiles)
features = {"descriptors": descriptors, "morgan": morgan}
print("Descriptor shape:", descriptors.shape, "Morgan shape:", morgan.shape)
print("Dense feature storage (MB):", round((descriptors.nbytes + morgan.nbytes) / 1e6, 2))
descriptor_names = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
                    "RingCount", "NumRotatableBonds", "FractionCSP3"]
display(pd.DataFrame(descriptors[train[:5]], columns=descriptor_names))
plt.hist(frame.iloc[train].exp, bins=20, edgecolor="white")
plt.xlabel("Training logD at pH 7.4")
plt.ylabel("Training molecules")
plt.title("Training distribution only; no test-driven tuning")
plt.show()

## 7. Train and select without looking at test errors

First fit a constant: the training-label mean. It asks whether using the
molecular input adds value. Ridge minimizes squared training error plus
alpha times the squared coefficient norm; the intercept is unpenalized.
Larger alpha shrinks coefficients more. Validation RMSE selects among
`[0.1, 1, 10, 100]`; exact ties select the larger alpha.

Descriptor scaling centers and scales. Bit scaling uses training variances
without centering. Thus this compares two complete recipes, not an isolated
causal effect of representation. LSQR uses tolerance 1e-6 and at most 10,000
iterations. Stop on convergence failure. Keep the selected training fit;
do not refit on train plus validation. This cell performs eight small Ridge
fits on CPU. No result is known until you run it.

**Before running:** predict how a constant baseline might beat a model.
More features can increase variance or extrapolation error.

In [ ]:
def score(y, prediction):
    """RMSE and MAE in logD units; reject accidental array broadcasting."""
    y, prediction = np.asarray(y, dtype=float), np.asarray(prediction, dtype=float)
    if y.ndim != 1 or prediction.shape != y.shape or not len(y):
        raise ValueError("Expected equally sized, nonempty one-dimensional arrays")
    if not np.isfinite(y).all() or not np.isfinite(prediction).all():
        raise ValueError("Labels and predictions must be finite")
    error = prediction - y
    return {"rmse": float(np.sqrt(np.mean(error**2))), "mae": float(np.mean(np.abs(error)))}

def fit_candidates(x_train, y_train, x_validation, y_validation, center):
    """Train-only scaling/fitting; validation-only alpha choice. No test argument."""
    candidates, rows = [], []
    for alpha in [0.1, 1.0, 10.0, 100.0]:
        model = make_pipeline(
            StandardScaler(with_mean=center),
            Ridge(alpha=alpha, solver="lsqr", tol=1e-6, max_iter=10000, fit_intercept=True),
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(x_train, y_train)
        if np.any(np.asarray(model.named_steps["ridge"].n_iter_) >= 10000):
            raise RuntimeError("Ridge reached max_iter; report convergence failure")
        metrics = score(y_validation, model.predict(x_validation))
        rows.append({"alpha": alpha, "validation_rmse": metrics["rmse"]})
        candidates.append(model)
    selected = min(range(len(rows)), key=lambda i: (rows[i]["validation_rmse"], -rows[i]["alpha"]))
    return candidates[selected], pd.DataFrame(rows)


y_train = frame.iloc[train].exp.to_numpy()
y_validation = frame.iloc[validation].exp.to_numpy()
train_mean = float(y_train.mean())
selected_models, validation_scores, selected_alpha = {}, {}, {}
for name, matrix in features.items():
    model, table = fit_candidates(
        matrix[train], y_train, matrix[validation], y_validation,
        center=(name == "descriptors"),
    )
    selected_models[name] = model
    validation_scores[name] = table.to_dict(orient="records")
    selected_alpha[name] = float(model.named_steps["ridge"].alpha)
    display(table.assign(representation=name))
print("Selected alpha:", selected_alpha)
print("Constant validation scores:",
      score(y_validation, np.full(len(validation), train_mean)))

## 8. Evaluate once, keeping every test molecule

RMSE is the square root of average squared error; MAE is average absolute
error. Both use logD units. RMSE weights large errors more strongly.
For errors `[1, -2]`, MAE = 1.5 and RMSE = sqrt(2.5), about 1.581.
All three methods below use the same test molecules. Do not remove a model
because it is worse than the constant, or select a new alpha here.

Each plotted point is one test molecule, not an independent assay repeat.
The dashed diagonal is perfect agreement; the residual line is zero error.
Plots show all held-out records without error bars or post-hoc filtering.

In [ ]:
y_test = frame.iloc[test].exp.to_numpy()
predictions = {"train_mean": np.full(len(test), train_mean)}
for name, model in selected_models.items():
    predictions[name] = model.predict(features[name][test])
test_metrics = {name: score(y_test, pred) for name, pred in predictions.items()}
display(pd.DataFrame(test_metrics).T.rename_axis("method"))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
low = min(y_test.min(), *(values.min() for values in predictions.values()))
high = max(y_test.max(), *(values.max() for values in predictions.values()))
for name, values in predictions.items():
    axes[0].scatter(y_test, values, s=16, alpha=0.55, label=name)
    axes[1].scatter(y_test, values - y_test, s=16, alpha=0.55, label=name)
axes[0].plot([low, high], [low, high], "k--", linewidth=1)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set(xlabel="Measured logD", ylabel="Predicted logD")
axes[1].set(xlabel="Measured logD", ylabel="Prediction minus measurement")
axes[0].legend()
fig.suptitle(f"All {len(test)} test molecules; fixed selected models")
fig.savefig(OUTPUT_DIR / "prediction-errors.png", dpi=140)
plt.show()

## 9. Compare paired errors, not just the smallest number

For each named pair, report RMSE(left) minus RMSE(right). Negative means
lower error for the left method. Resample whole test-scaffold groups with
replacement, preserving all their molecules and pairing predictions.
Recompute molecule-weighted RMSE differences in 2,000 resamples with seed
20260909. The 2.5th/97.5th percentiles form a conditional 95% bootstrap interval.

These are fixed-model, fixed-test-population uncertainty summaries. They
assume scaffold groups are exchangeable; they omit training-set variation,
measurement uncertainty and cross-assay shift. Three intervals are shown,
without multiple-comparison correction or significance claims. With fewer
than two groups the interval is not estimable. A narrow interval is not
evidence that this is a SOTA model or that the assay is unbiased.

In [ ]:
def paired_interval(y, left, right, groups):
    """Fixed-model RMSE(left)-RMSE(right), paired whole-scaffold bootstrap.

    Uniformly sample G groups G times with replacement, retain all their rows,
    and recompute molecule-weighted RMSE differences. Not a prediction interval
    or a claim of independent assay replication; no multiplicity adjustment.
    """
    difference = score(y, left)["rmse"] - score(y, right)["rmse"]
    groups = np.asarray(groups)
    if groups.ndim != 1 or len(groups) != len(y) or pd.isna(groups).any():
        raise ValueError("One nonmissing scaffold key is required per prediction")
    keys, inverse = np.unique(groups, return_inverse=True)
    result = {
        "difference": difference,
        "ci95": None,
        "groups": len(keys),
        "rows": len(y),
        "resamples": 0,
    }
    if len(keys) < 2:
        return result
    counts = np.bincount(inverse)
    left_sse = np.bincount(inverse, weights=(np.asarray(left) - np.asarray(y)) ** 2)
    right_sse = np.bincount(inverse, weights=(np.asarray(right) - np.asarray(y)) ** 2)
    rng = np.random.default_rng(20260909)
    differences = []
    for _ in range(2000):
        draw = rng.integers(0, len(keys), size=len(keys))
        n = counts[draw].sum()
        differences.append(np.sqrt(left_sse[draw].sum() / n) - np.sqrt(right_sse[draw].sum() / n))
    result["ci95"] = np.percentile(differences, [2.5, 97.5]).tolist()
    result["resamples"] = 2000
    return result


intervals = {}
for left, right in [("descriptors", "train_mean"), ("morgan", "train_mean"),
                    ("descriptors", "morgan")]:
    intervals[f"{left}-minus-{right}"] = paired_interval(
        y_test, predictions[left], predictions[right], frame.iloc[test].scaffold,
    )
display(pd.DataFrame(intervals).T)

## 10. Explain a failure without changing the test

These diagnostics were named before training: endpoint records, disconnected
components, scaffold-group size and nearest-training Morgan/Tanimoto similarity.
The original intake had one value at −1.5 and four at 4.5. Missing relation
qualifiers do not establish that those boundary measurements are uncensored.
Similarity is descriptive; it is not another training feature in this recipe.

Inspect the tables, including empty slices. A pattern suggests a hypothesis,
not a correction to apply retroactively. Missing batch/replicate columns
prevent attributing errors to assay noise alone.

In [ ]:
error_table = frame.iloc[test][["CMPD_CHEMBLID", "smiles", "scaffold", "exp"]].copy()
for name, values in predictions.items():
    error_table[name] = values
    error_table[f"{name}_absolute_error"] = np.abs(values - y_test)
error_table["range_endpoint"] = error_table.exp.isin([-1.5, 4.5])
error_table["multicomponent"] = [
    len(Chem.GetMolFrags(Chem.MolFromSmiles(s))) > 1 for s in error_table.smiles
]
error_table["scaffold_size"] = error_table.groupby("scaffold").scaffold.transform("size")
train_fingerprints = [fingerprints[i] for i in train]
error_table["nearest_train_similarity"] = [
    max(DataStructs.BulkTanimotoSimilarity(fingerprints[i], train_fingerprints))
    for i in test
]
display(error_table.head(10))
display(error_table.groupby("range_endpoint").agg(
    molecules=("exp", "size"), descriptor_mae=("descriptors_absolute_error", "mean"),
    morgan_mae=("morgan_absolute_error", "mean"),
))
display(error_table.loc[error_table.multicomponent])
error_table.to_csv(OUTPUT_DIR / "test-predictions.csv", index=False)

## 11. Save a report you can explain

The files contain your actual metrics, paired intervals, selections, split
membership and predictions. No pretrained weights are required. Download
the output folder's files using Colab's Files sidebar, and save the notebook
with your observations. Do not upload sensitive data or another person's files.

Write five sentences: the question and endpoint; data and split counts;
validation-selected recipes; test errors and conditional differences;
one limitation and a new-data check that could change your conclusion.
Copy numbers from your run, not an expected answer. Preserve negative results.

In [ ]:
summary = {
    "data_kind": DATA_KIND,
    "endpoint": "logD at pH 7.4",
    "counts": counts,
    "selected_alpha": selected_alpha,
    "validation_scores": validation_scores,
    "test_metrics": test_metrics,
    "paired_scaffold_intervals": intervals,
    "excluded_rows": 0,
    "interpretation": ("Student exercise; conditional fixed-model intervals; "
                       "not a SOTA or cross-assay claim"),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
print("Your outputs:", OUTPUT_DIR)
print("Files:", sorted(path.name for path in OUTPUT_DIR.iterdir()))

## 12. Independent practice and answer checks

1. **Before opening the validation table**, predict which alpha should shrink
   coefficients most. Inspect the selected model's `ridge.coef_` and the
   training scaler's `mean_`. Explain why neither used test labels.
2. Compute RMSE and MAE by hand for three rows of your exported predictions.
   Compare with `score` on those same three rows, not the full test metric.
3. In a separate copy, change only held-out `exp` values after keeping the
   same membership. Features, selected alpha and predictions should remain
   unchanged; reported errors should change. This is an engineering check
   with artificial labels, not an improved empirical result.
4. If Morgan loses to the constant, propose one validation-only investigation
   and one genuinely new measurement. Do not rerank recipes on this test.

**Answer checks:** 100 is the strongest candidate penalty; the pipeline fits
its scaler only inside `fit_candidates` using training rows. A label change
cannot alter a structure-only fingerprint. An error pattern alone does not
identify its cause. For stronger evidence, collect a new assay/batch or an
independently designed evaluation, rather than repeatedly reading this test.